# Extended Data Fig. 1: Segger's training objective

A composite metric-learning objective shapes the joint transcript–cell embedding.
Panels a and b are schematics of the cell (L_ce), segmentation (L_sg) and transcript (L_tx) losses, defined in `segger.models`; this notebook draws the embedding over training (c) and the loss ablations (d) on the healthy colon section of Fig. 5.

| | |
|---|---|
| Data | Xenium healthy colon section; the annotated Segger cells written by the Fig. 5 notebook |
| Compute | CUDA GPU; six Segger trainings of 20 epochs. Models, checkpoints and embeddings are cached in `results/ed_figure_1`. Run `figure_5_xenium_healthy_colon.ipynb` first. |
| Output | `results/ed_figure_1/panel_{c,d}.{pdf,png}` |

In [ ]:
import json
import logging
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)  # deprecation notices raised on import

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import scanpy as sc
import seaborn as sns
import torch
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.plugins.environments import SLURMEnvironment
from scipy.cluster.hierarchy import leaves_list, linkage
from segger.data import ISTDataModule, ISTSegmentationWriter
from segger.io import get_preprocessor
from segger.models import LitISTEncoder
from tqdm.auto import tqdm
from umap import AlignedUMAP

import sg_utils as sg

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
for name in ["pytorch_lightning", "lightning.pytorch", "lightning_fabric"]:
    logging.getLogger(name).setLevel(logging.ERROR)
SLURMEnvironment.detect = lambda: False  # no SLURM requeueing inside a notebook
ASSETS = Path(sg.pl.__file__).parent / "assets"
plt.style.use(ASSETS / "default.mplstyle")
plt.rcParams["savefig.bbox"] = "tight"
palettes = json.loads((ASSETS / "palettes.json").read_text())
cell_type_palette = palettes["Cell Type"]
compartment_palette = palettes["Compartment"]

In [ ]:
DATA_DIR = Path("../data")
FIG5_DIR = Path("../results/figure_5")  # annotated Segger cells of the Fig. 5 notebook
OUT_DIR = Path("../results/ed_figure_1")
N_EPOCHS = 20
SEED = 42

## Download the data

In [ ]:
paths = sg.datasets.fetch("xenium_healthy_colon", DATA_DIR, files=["xenium"])
XENIUM_DIR = paths["xenium"]
annotated = sc.read_h5ad(FIG5_DIR / "xenium_healthy_colon_segger.h5ad")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Ablation models

Each ablation zeroes one or more loss weights.
Without the cell loss, cell nodes are represented by nuclear morphology instead of expression principal components; without the transcript loss, the gene embedding stays at its initialization.
Models are trained only when their segmentation is not in `OUT_DIR/ablations`.

In [ ]:
ABLATIONS = {
    "all_components": {"data_module": {}, "model": {}},
    "no_segmentation_loss": {"data_module": {}, "model": {"sg_weight_start": 0.0, "sg_weight_end": 0.0}},
    "no_cell_transcript_losses": {
        "data_module": {"cells_representation_mode": "morphology"},
        "model": {
            "bd_weight_start": 0.0, "bd_weight_end": 0.0, "tx_weight_start": 0.0, "tx_weight_end": 0.0,
            "update_gene_embedding": False,
        },
    },
    "no_cell_loss": {
        "data_module": {"cells_representation_mode": "morphology"},
        "model": {"bd_weight_start": 0.0, "bd_weight_end": 0.0},
    },
    "no_transcript_loss": {
        "data_module": {},
        "model": {"tx_weight_start": 0.0, "tx_weight_end": 0.0, "update_gene_embedding": False},
    },
}
ABLATION_LABELS = {
    "all_components": "All Components",
    "no_segmentation_loss": "-Segmentation Loss",
    "no_cell_transcript_losses": "-Cell / Transcript Losses",
    "no_cell_loss": "-Cell Loss",
    "no_transcript_loss": "-Transcript Loss",
}


def train_segger(outs_dir, data_module_kwargs, model_kwargs, callbacks=()):
    """Train Segger on the colon section and write its segmentation to ``outs_dir``."""
    outs_dir = Path(outs_dir)
    outs_dir.mkdir(parents=True, exist_ok=True)
    data_module = ISTDataModule(
        XENIUM_DIR, prediction_graph_mode="cell", prediction_graph_buffer_ratio=0.2,
        tiling_nodes_per_tile=50_000, edges_per_batch=1_000_000, **data_module_kwargs,
    )
    model = LitISTEncoder(n_genes=data_module.ad.shape[1], in_channels=data_module.cells_embedding_size, **model_kwargs)
    writer = ISTSegmentationWriter(outs_dir)
    trainer = Trainer(
        logger=CSVLogger(outs_dir), max_epochs=N_EPOCHS, reload_dataloaders_every_n_epochs=1,
        callbacks=[writer, *callbacks], enable_progress_bar=True, enable_model_summary=False,
    )
    trainer.fit(model=model, datamodule=data_module)
    predictions = trainer.predict(model=model, datamodule=data_module)
    writer.write_on_epoch_end(trainer=trainer, pl_module=model, predictions=predictions, batch_indices=[])
    return data_module


for name, kwargs in ABLATIONS.items():
    outs_dir = OUT_DIR / "ablations" / name
    if (outs_dir / "segger_segmentation.parquet").exists():
        continue
    train_segger(outs_dir, kwargs["data_module"], kwargs["model"])
    torch.cuda.empty_cache()

## Panel d: marker similarity per ablation

Mean similarity between the transcripts of a marker gene and the cells they were assigned to, for the cells of each type group (`cell_type_level_2`); five markers per group.
Rows and columns are ordered by clustering the full-objective matrix; the colour scale spans one segmentation-loss margin (0.4) below the maximum.

In [ ]:
group_key = "cell_type_level_2"
N_MARKERS = 5
SEGMENTATION_MARGIN = 0.4  # LitISTEncoder default

sc.tl.rank_genes_groups(annotated, groupby=group_key, use_raw=False)
markers = (
    sc.get.rank_genes_groups_df(annotated, group=None)
    .sort_values(["names", "logfoldchanges"])
    .groupby("names")
    .last()
    .sort_values(["group", "logfoldchanges"])
    .groupby("group")
    .tail(N_MARKERS)
)
transcripts = get_preprocessor(XENIUM_DIR).transcripts.filter(pl.col("feature_name").is_in(markers.index.to_numpy()))
cell_types = pl.from_pandas(annotated.obs[[group_key]], include_index=True).rename({"index": "segger_cell_id"})

similarity = dict()
for name in ABLATIONS:
    segmentation = pl.read_parquet(OUT_DIR / "ablations" / name / "segger_segmentation.parquet")
    similarity[name] = (
        transcripts
        .join(segmentation, on="row_index")
        .join(cell_types, on="segger_cell_id", how="inner")
        .group_by([group_key, "feature_name"])
        .agg(pl.col("segger_similarity").mean())
        .pivot(on="feature_name", index=group_key, values="segger_similarity")
        .fill_null(0)
        .to_pandas()
        .set_index(group_key)
    )

reference = similarity["all_components"]
group_order = reference.index[leaves_list(linkage(reference, method="single"))]
gene_order = []
for group in group_order:
    genes = markers.loc[markers["group"] == group].index
    gene_order += genes[leaves_list(linkage(reference.loc[:, genes].T, method="weighted"))].tolist()
vmax = max(frame.to_numpy().max() for frame in similarity.values())
vmin = vmax - SEGMENTATION_MARGIN

fig, axes = plt.subplots(1, len(ABLATIONS), figsize=(12.5, 2), sharey=True)
for ax, name in zip(axes, ABLATIONS):
    sns.heatmap(
        similarity[name].loc[group_order, gene_order], vmin=vmin, vmax=vmax, cmap="rocket", cbar=None,
        xticklabels=True, ax=ax,
    )
    ax.set_aspect(N_MARKERS)
    ax.set_title(ABLATION_LABELS[name])
    ax.set_xlabel("")
    ax.set_ylabel("")
sg.pl.save_panel(fig, OUT_DIR / "panel_d")

## Panel c: joint embedding over training

The full objective is retrained with a checkpoint every 50 steps.
A fixed sample of 5,000 annotated cells and five assigned transcripts per cell is embedded by every checkpoint, and the checkpoints are embedded jointly with `AlignedUMAP` (each cell linked to itself across checkpoints).
Cells are coloured by cell type, transcripts by the compartment of their cell.

In [ ]:
trajectory_dir = OUT_DIR / "trajectory"
checkpoint_dir = trajectory_dir / "lightning_logs/version_0/checkpoints"
if not checkpoint_dir.exists():
    train_segger(
        trajectory_dir, data_module_kwargs={}, model_kwargs={},
        callbacks=[ModelCheckpoint(every_n_train_steps=50, save_top_k=-1, save_on_train_epoch_end=False)],
    )
    torch.cuda.empty_cache()
checkpoints = {int(path.stem.split("=")[-1]): path for path in sorted(checkpoint_dir.glob("*.ckpt"))}

In [ ]:
N_CELLS = 5_000
N_TRANSCRIPTS_PER_CELL = 5
TRAJECTORY_STEPS = [0, 3000]

path_trajectory = OUT_DIR / "trajectory_embeddings.pt"
path_segmentation = OUT_DIR / "ablations/all_components/segger_segmentation.parquet"
if path_trajectory.exists():
    trajectory = torch.load(path_trajectory)
else:
    data_module = ISTDataModule.load_from_checkpoint(
        next(iter(checkpoints.values())), tiling_nodes_per_tile=500_000, edges_per_batch=10_000_000,
    )
    data_module.setup("predict")
    segmentation = pl.read_parquet(path_segmentation)
    obs = data_module.ad.obs
    keep = obs["filtered"] & obs["cell_id"].isin(annotated.obs.index)
    cells = obs.loc[keep, ["cell_id", "cell_encoding"]].sample(N_CELLS, random_state=SEED)
    sampled = (
        segmentation
        .filter(
            pl.col("segger_similarity").gt(pl.col("similarity_threshold")),
            pl.col("segger_cell_id").is_in(cells["cell_id"].to_list()),
        )
        .sort("segger_similarity", descending=True)
        .group_by("segger_cell_id")
        .head(N_TRANSCRIPTS_PER_CELL)
    )
    cell_index = torch.tensor(cells["cell_encoding"].values).cuda()
    transcript_index = torch.tensor(sampled["row_index"].to_numpy()).cuda()

    trajectory = dict(steps=[], cells=[], transcripts=[])
    with torch.no_grad():
        for step, path in tqdm(sorted(checkpoints.items())):
            model = LitISTEncoder.load_from_checkpoint(path).cuda().eval()
            batches, indices = dict(bd=[], tx=[]), dict(bd=[], tx=[])
            for batch in data_module.predict_dataloader():
                batch = batch.to("cuda")
                masks = dict(
                    bd=torch.isin(batch["bd"]["index"], cell_index),
                    tx=torch.isin(batch["tx"]["index"], transcript_index),
                )
                if not masks["bd"].any() or not masks["tx"].any():
                    continue
                embeddings = model.forward(batch)
                for node_type, mask in masks.items():
                    batches[node_type].append(embeddings[node_type][mask].cpu())
                    indices[node_type].append(batch[node_type]["index"][mask].cpu())
            trajectory["steps"].append(step)
            trajectory["cells"].append(torch.cat(batches["bd"]))
            trajectory["transcripts"].append(torch.cat(batches["tx"]))
            del model
            torch.cuda.empty_cache()

    encoding_to_id = data_module.ad.obs.set_index("cell_encoding")["cell_id"]
    trajectory["cell_ids"] = encoding_to_id.loc[torch.cat(indices["bd"]).numpy()].to_numpy()
    trajectory["transcript_cell_ids"] = (
        pl.DataFrame({"row_index": torch.cat(indices["tx"]).numpy()})
        .join(segmentation, on="row_index", how="left")["segger_cell_id"]
        .to_numpy()
    )
    torch.save(trajectory, path_trajectory)

In [ ]:
path_umap = OUT_DIR / "trajectory_umap.pt"
if path_umap.exists():
    coordinates = torch.load(path_umap)
else:
    cells = [frame.numpy() for frame in trajectory["cells"]]
    rng = np.random.default_rng(SEED)
    subset = rng.choice(cells[0].shape[0], cells[0].shape[0] // 10, False)
    relations = [{i: i for i in range(len(subset))}] * (len(cells) - 1)
    mapper = AlignedUMAP(
        n_neighbors=30, n_components=2, metric="cosine", alignment_regularisation=0.1, alignment_window_size=3,
        n_epochs=500, random_state=SEED,
    )
    mapper.fit([frame[subset] for frame in cells], relations=relations)
    coordinates = {
        key: [m.transform(frame.numpy()) for m, frame in zip(mapper.mappers_, trajectory[key])]
        for key in ["cells", "transcripts"]
    }
    torch.save(coordinates, path_umap)

In [ ]:
cell_labels = annotated.obs.loc[trajectory["cell_ids"], "cell_type_level_1"].to_numpy()
transcript_labels = annotated.obs.reindex(trajectory["transcript_cell_ids"])["cell_type_compartment"].to_numpy()

fig, axes = plt.subplots(1, len(TRAJECTORY_STEPS), figsize=(4, 2))
for ax, step in zip(axes, TRAJECTORY_STEPS):
    frame = trajectory["steps"].index(step)
    for key, labels, palette in [
        ("transcripts", transcript_labels, compartment_palette),
        ("cells", cell_labels, cell_type_palette),
    ]:
        xy = coordinates[key][frame]
        ax.scatter(
            xy[:, 0], xy[:, 1], s=0.5, linewidths=0, alpha=0.5,
            c=[palette.get(label, "0.5") for label in labels], rasterized=True,
        )
    ax.set_title(f"Step {step:,}")
    ax.set_aspect("equal")
    ax.axis("off")
sg.pl.save_panel(fig, OUT_DIR / "panel_c")

## Software versions

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    display(sc.logging.print_header())